# Basic Pipeline

End-to-end simulation: light source → surface → scattering → optics → detector → analysis.

In [ ]:
import numpy as np
from optical_metrology.illumination import Laser, GaussianBeamProfile
from optical_metrology.surface import RoughSurface, Material
from optical_metrology.scattering import LambertianScattering
from optical_metrology.optics import OpticalSystem, GaussianPSF, OpticalPropagator
from optical_metrology.detector import CMOSDetector
from optical_metrology.analysis import HistogramAnalyzer, ImageAnalyzer

In [ ]:
laser = Laser(wavelength=532e-9, power=5e-3, beam_profile=GaussianBeamProfile(w0=2.0))
laser.propagation_direction = np.array([0.0, 0.0, -1.0])
lf = laser.generate_light_field(shape=(32, 32), spacing=0.5)
print(f"Intensity: {lf.intensity.min():.3g} - {lf.intensity.max():.3g}")

In [ ]:
surface = RoughSurface((32, 32), sigma=4.0, amplitude=0.3, material=Material("silicon"))
print(f"Roughness: {surface.roughness:.4g}")

In [ ]:
scattered = LambertianScattering(albedo=0.7).evaluate(
    lf, surface, view_direction=np.array([0.0, 0.0, 1.0])
)
print(f"Radiance: {scattered.radiance.min():.3g} - {scattered.radiance.max():.3g}")

In [ ]:
optics = OpticalSystem(focal_length=0.05, aperture_diameter=0.008, wavelength=532e-9)
sensor = OpticalPropagator(GaussianPSF(sigma=1.0)).propagate(scattered, optics)
print(f"Irradiance: {sensor.irradiance.min():.3g} - {sensor.irradiance.max():.3g}")

In [ ]:
image = CMOSDetector(exposure_time=1e-5, gain=1.0).capture(sensor)
print(f"Digital image: {image.pixels.shape}, {image.pixels.min()}-{image.pixels.max()} ADU")

In [ ]:
report = ImageAnalyzer(modules=[HistogramAnalyzer()]).analyze(image)
for k, v in report.measurements.items():
    print(f"  {k}: {v:.4g}")

In [ ]:
print(image.visualize(max_width=48))